In [1]:
from os import PathLike
from hdfs import InsecureClient
from pyspark.sql import SparkSession, Row
from delta import *
from pyspark.sql.types import (LongType, StringType, StructField, StructType,BooleanType, ArrayType, IntegerType, DoubleType)
from pyspark.sql.functions import (col, trim, when, greatest, expr)
from pyspark.sql.functions import col, when, regexp_replace, lit, coalesce
from pyspark.sql.types import StringType, DoubleType, LongType

In [2]:
# warehouse_location points to the default location for managed databases and tables
warehouse_location = 'hdfs://hdfs-nn:9000/demo'

builder = (
    SparkSession.builder
    .appName("Python Spark SQL Hive integration example")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:1.0.0")
    .enableHiveSupport()
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/boxoffice.csv"
# se tiver outro nome no HDFS, ajusta aqui

df_bronze = (
    spark.read
        .option("header", "true")     # tem cabeçalho
        .option("delimiter", ",")     # CSV normal
        .csv(hdfs_path)
)

df_bronze.printSchema()
df_bronze.show(5)

root
 |-- Year: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Gross: string (nullable = true)

+----+--------------------+------------+
|Year|               Title|       Gross|
+----+--------------------+------------+
|1984|   Beverly Hills Cop|$234,760,478|
|1984|        Ghostbusters|$229,376,332|
|1984|Indiana Jones and...|$179,876,727|
|1984|            Gremlins|$148,171,538|
|1984|      The Karate Kid| $90,817,155|
+----+--------------------+------------+
only showing top 5 rows



In [4]:
df_silver = (
    df_bronze
        .withColumn("Title", trim(col("Title")))
        .withColumn("Year", col("Year").cast(IntegerType()))
        .withColumn(
            "gross_clean",
            when(
                col("Gross").isNotNull() & (col("Gross") != ""),
                regexp_replace(
                    col("Gross").cast(StringType()),
                    r"[^0-9]",
                    ""
                ).cast(LongType())
            ).otherwise(lit(None).cast(LongType()))
        )
        .drop("Gross")
)

df_silver.printSchema()

df_silver_display = df_silver.select([
    coalesce(col(c).cast(StringType()), lit("null")).alias(c)
    for c in df_silver.columns
])

root
 |-- Year: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- gross_clean: long (nullable = true)



In [5]:
df_silver_display.select("Year", "Title", "gross_clean").show(20, truncate=False)

+----+-------------------------------------------------+-----------+
|Year|Title                                            |gross_clean|
+----+-------------------------------------------------+-----------+
|1984|Beverly Hills Cop                                |234760478  |
|1984|Ghostbusters                                     |229376332  |
|1984|Indiana Jones and the Temple of Doom             |179876727  |
|1984|Gremlins                                         |148171538  |
|1984|The Karate Kid                                   |90817155   |
|1984|Police Academy                                   |81198894   |
|1984|Footloose                                        |80038626   |
|1984|Romancing the Stone                              |76572547   |
|1984|Star Trek III: The Search for Spock              |76471046   |
|1984|Splash                                           |69821334   |
|1984|Purple Rain                                      |68393249   |
|1984|Amadeus                     

In [6]:
SILVER_PATH = "hdfs://hdfs-nn:9000/demo/silver/boxoffice_silver"

(
    df_silver
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("Year")
        .save(SILVER_PATH)
)

In [7]:
spark.sql(
    """
    SHOW DATABASES
    """
).show()

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+



In [8]:
spark.sql(
    """
    DROP TABLE IF EXISTS demo.boxoffice_silver
    """
)

DataFrame[]

In [9]:
spark.sql(
    """
    CREATE EXTERNAL TABLE demo.boxoffice_silver (
        Title        STRING,
        Year         INT,
        gross_clean  BIGINT
    )
    USING DELTA
    PARTITIONED BY (Year)
    LOCATION 'hdfs://hdfs-nn:9000/demo/silver/boxoffice_silver'
    """
)

DataFrame[]

In [10]:
spark.sql("SHOW TABLES FROM demo").show()

spark.sql(
    """
    SELECT Year, Title, gross_clean
    FROM demo.boxoffice_silver
    ORDER BY Year, Title
    """
).show(20, truncate=False)

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|     demo|boxoffice_silver|      false|
|     demo| deltalake_table|      false|
+---------+----------------+-----------+

+----+---------------------------------+-----------+
|Year|Title                            |gross_clean|
+----+---------------------------------+-----------+
|1984|1984                             |8430492    |
|1984|2010                             |40400657   |
|1984|A Christmas Story 1984 Re-release|1311065    |
|1984|A Nightmare on Elm Street        |25507443   |
|1984|A Passage to India               |27187653   |
|1984|A Soldier's Story                |21821347   |
|1984|A Sunday in the Country          |2411143    |
|1984|Against All Odds                 |21689062   |
|1984|All of Me                        |36403064   |
|1984|Alphabet City                    |7035585    |
|1984|Amadeus                          |51565096   |
|1984|Ameri

In [11]:
spark.sql("SELECT * FROM demo.boxoffice_silver").toPandas()

,Year,Title,gross_clean
0,2023,Barbie,1445638421
1,2023,The Super Mario Bros. Movie,1362034072
2,2023,Oppenheimer,972444774
3,2023,Guardians of the Galaxy Vol. 3,845555777
4,2023,Fast X,704875015
...,...,...,...
8140,1984,Heartbreakers,148750
8141,1984,Crackers,129268
8142,1984,Repo Man,129000
8143,1984,Eureka,123572


In [12]:
spark.stop()